<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

# Python for Algorithmic Trading

&copy; Dr. Yves J. Hilpisch | The Python Quants GmbH

http://tpq.io | [training@tpq.io](mailto:trainin@tpq.io) | [@dyjh](http://twitter.com/dyjh)

### Please use the "Python 3.10, Numpy 1.26.4 Pandas TA" kernel.

In [ ]:
!git clone https://github.com/tpq-classes/python_for_algo_trading_practice.git
import sys
sys.path.append('python_for_algo_trading_practice')


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
from pylab import plt
%config InlineBackend.figure_format = 'svg'

In [ ]:
import warnings as w
w.simplefilter('ignore')

## API Connection

In [ ]:
import tpqoa  # wrapper package for the Oanda v20 REST API

In [ ]:
api = tpqoa.tpqoa('../../../oanda.cfg')  # adjust path as needed

In [ ]:
api.get_instruments()[-7:]

## Historical Data

In [ ]:
sym = 'BCO_USD'

In [ ]:
instrument = sym
start = '2022-06-01'
end = '2023-07-06'
granularity = 'H1'
price = 'M'

In [ ]:
%time raw = api.get_history(instrument, start, end, granularity, price)[list('ohlc')]

In [ ]:
raw.info()

In [ ]:
# change the DatetimeIndex from left to right label
# for every Timestamp in the DatetimeIndex the values are known afterwards
raw.index += pd.Timedelta(hours=1)  # adjustment needs to be in line with granularity

In [ ]:
raw.head()

In [ ]:
raw.tail()

## Visualization

In [ ]:
import pandas_ta as ta
import mplfinance as mpf
plt.style.use('seaborn-v0_8')

In [ ]:
raw.columns = ['Open', 'High', 'Low', 'Close']

In [ ]:
mpf.plot(raw.iloc[-100:]);

In [ ]:
mpf.plot(raw.iloc[-100:], type='candle', style='charles');

In [ ]:
mpf.plot(raw.iloc[-100:], type='candle', style='charles', mav=(3, 10));

In [ ]:
raw.ta.sma(length=3)

In [ ]:
raw.tail()

In [ ]:
raw.ta.sma(length=3, append=True)

In [ ]:
raw.ta.sma(length=10, append=True)

In [ ]:
raw.tail()

In [ ]:
raw[['Close', 'SMA_3', 'SMA_10']].iloc[-150:].plot();

In [ ]:
ha = raw.ta.ha()

In [ ]:
ha.head()

In [ ]:
ha.columns = ['Open', 'High', 'Low', 'Close']

In [ ]:
mpf.plot(ha.iloc[-75:], type='candle', style='charles')

In [ ]:
bb = raw.ta.bbands()
bb.iloc[:, :3].iloc[-250:].plot();

## Rolling Backtest

... of a **OLS-based strategy**.

In [ ]:
data = pd.DataFrame(raw['Close'])

In [ ]:
data.columns = [sym]

In [ ]:
data.plot();

In [ ]:
data['r'] = np.log(data[sym] / data[sym].shift(1))

In [ ]:
lags = 15
cols = list()
for lag in range(1, lags+1):
    col = f'lag_{lag}'
    data[col] = data['r'].shift(lag)
    cols.append(col)

In [ ]:
data.head()

In [ ]:
data.dropna(inplace=True)

In [ ]:
data['p'] = 0.0

In [ ]:
n = 750
start = n

In [ ]:
len(data) - n  # number of regressions and single predictions

In [ ]:
%%time
for bar in range(start, len(data)):
    # increasing interval
    # reg = np.linalg.lstsq(data[cols].iloc[:bar], data['r'].iloc[:bar], rcond=-1)[0]
    # fixed interval
    reg = np.linalg.lstsq(data[cols].iloc[bar-n:bar], data['r'].iloc[bar-n:bar], rcond=-1)[0]
    data.loc[data.index[bar], 'p'] = np.sign(np.dot(data[cols].iloc[bar], reg))

In [ ]:
# data.head()

In [ ]:
# data.tail()

In [ ]:
sum(np.sign(data['r']) == data['p']) / len(data)

In [ ]:
data['s'] = data['p'] * data['r']

In [ ]:
data[['r', 's']].iloc[start:].sum().apply(np.exp)

In [ ]:
data[['r', 's']].iloc[start:].cumsum().apply(np.exp).plot();

## Strategy Metrics

In [ ]:
import quantstats as qs

In [ ]:
qs.reports.basic(data['s'].iloc[start:], benchmark=data['r'].iloc[start:])

<img src="http://hilpisch.com/tpq_logo.png" alt="The Python Quants" width="35%" align="right" border="0"><br>

<a href="http://tpq.io" target="_blank">http://tpq.io</a> | <a href="http://twitter.com/dyjh" target="_blank">@dyjh</a> | <a href="mailto:training@tpq.io">training@tpq.io</a>